In [31]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [32]:
data = np.load("pems04.npz")

traffic = data["data"][:,:,2]

print(traffic.shape)

(16992, 307)


In [33]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

print(
    traffic_scaled.shape
)

(16992, 307)


In [34]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    -
    sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(16980, 12, 307)
(16980, 307)


In [35]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape)
print(X_test.shape)

(13584, 12, 307)
(3396, 12, 307)


In [36]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [37]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [38]:
class LSTMModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=307,
            hidden_size=64,
            batch_first=True
        )

        self.fc = nn.Linear(
            64,
            307
        )

    def forward(self,x):

        out, _ = self.lstm(x)

        out = out[:,-1,:]

        out = self.fc(out)

        return out

In [39]:
import time

train_start = time.time()

In [40]:
model = LSTMModel()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}",
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.043851
Epoch 2/30 Loss: 0.008834
Epoch 3/30 Loss: 0.007448
Epoch 4/30 Loss: 0.006900
Epoch 5/30 Loss: 0.006433
Epoch 6/30 Loss: 0.006006
Epoch 7/30 Loss: 0.005612
Epoch 8/30 Loss: 0.005273
Epoch 9/30 Loss: 0.005032
Epoch 10/30 Loss: 0.004851
Epoch 11/30 Loss: 0.004697
Epoch 12/30 Loss: 0.004557
Epoch 13/30 Loss: 0.004432
Epoch 14/30 Loss: 0.004306
Epoch 15/30 Loss: 0.004193
Epoch 16/30 Loss: 0.004099
Epoch 17/30 Loss: 0.003995
Epoch 18/30 Loss: 0.003921
Epoch 19/30 Loss: 0.003833
Epoch 20/30 Loss: 0.003751
Epoch 21/30 Loss: 0.003674
Epoch 22/30 Loss: 0.003611
Epoch 23/30 Loss: 0.003553
Epoch 24/30 Loss: 0.003496
Epoch 25/30 Loss: 0.003434
Epoch 26/30 Loss: 0.003381
Epoch 27/30 Loss: 0.003336
Epoch 28/30 Loss: 0.003285
Epoch 29/30 Loss: 0.003233
Epoch 30/30 Loss: 0.003192


In [41]:
train_time = time.time() - train_start

print("Time Taken:", train_time)

Time Taken: 191.51547813415527


In [42]:
torch.save(
    model.state_dict(),
    "LSTM-PEMS04.pth"
)

In [43]:
model.eval()

infer_start = time.time()
with torch.no_grad():

    predictions = model(
        X_test
    )

predictions = predictions.numpy()
infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = y_test.numpy()

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

Infer Time: 0.09998798370361328
MAE: 0.039301105
RMSE: 0.06674184


In [44]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)
print("MAPE:", mape)
print("R2:", r2)

MAPE: 5792.055511474609
R2: 0.7881780216346878


In [45]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 115443
